<a href="https://colab.research.google.com/github/kosebaris1/MACHINE_LEARNING/blob/RNN/RNN_TextGenerator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
import numpy as np
import re
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

masal_metin = """
bir varmış bir yokmuş evvel zaman içinde kalbur saman içinde pireler berber develer tellal
iken ben annemin beşiğini tıngır mıngır sallar iken uzak bir köyde keloğlan adında bir çocuk
yaşarmış keloğlan yoksulmuş ama çok akıllıymış bir gün annesi ona demiş ki oğlum evde
yiyecek kalmadı ormana git biraz odun kes keloğlan baltasını alıp ormana gitmiş odun
keserken bir dev çıkagelmiş dev demiş ki bu ormanda odun kesmek yasak keloğlan korkmuş
ama hemen düşünmüş demiş ki ey dev ben senin yardımcın olurum bana izin ver dev bu
cevabı beğenmiş ve keloğlanı yanına almış günler geçmiş keloğlan devin evinde çalışmış bir
gün dev hastalanmış keloğlan ona bakmış ilaç getirmiş dev iyileşince çok sevinmiş demiş ki ey
keloğlan sen iyi bir çocuksun al bu altınları annenle mutlu yaşa keloğlan altınları almış
annesine dönmüş ve bir daha hiç yoksulluk çekmemişler gökten üç elma düşmüş biri anlatana
biri dinleyene biri de keloğlana
"""

print("TensorFlow version:", tf.__version__)


TensorFlow version: 2.19.0


In [13]:

text = masal_metin.lower()

text = re.sub(r'[^a-zçğıöşü ]+', ' ', text)

text = re.sub(r'\s+', ' ', text).strip()

print("Temizlenmiş metin:")
print(text)
print("\nToplam karakter sayısı:", len(text))

words = text.split(' ')
print("\nİlk 20 kelime:", words[:20])
print("Toplam kelime sayısı:", len(words))


Temizlenmiş metin:
bir varmış bir yokmuş evvel zaman içinde kalbur saman içinde pireler berber develer tellal iken ben annemin beşiğini tıngır mıngır sallar iken uzak bir köyde keloğlan adında bir çocuk yaşarmış keloğlan yoksulmuş ama çok akıllıymış bir gün annesi ona demiş ki oğlum evde yiyecek kalmadı ormana git biraz odun kes keloğlan baltasını alıp ormana gitmiş odun keserken bir dev çıkagelmiş dev demiş ki bu ormanda odun kesmek yasak keloğlan korkmuş ama hemen düşünmüş demiş ki ey dev ben senin yardımcın olurum bana izin ver dev bu cevabı beğenmiş ve keloğlanı yanına almış günler geçmiş keloğlan devin evinde çalışmış bir gün dev hastalanmış keloğlan ona bakmış ilaç getirmiş dev iyileşince çok sevinmiş demiş ki ey keloğlan sen iyi bir çocuksun al bu altınları annenle mutlu yaşa keloğlan altınları almış annesine dönmüş ve bir daha hiç yoksulluk çekmemişler gökten üç elma düşmüş biri anlatana biri dinleyene biri de keloğlana

Toplam karakter sayısı: 922

İlk 20 kelime: ['bir', 'varm

In [14]:
vocab = sorted(set(words))
vocab_size = len(vocab)
print("Vocab size:", vocab_size)

word_to_idx = {w: i for i, w in enumerate(vocab)}
idx_to_word = {i: w for w, i in word_to_idx.items()}


Vocab size: 103


In [15]:
seq_length = 5
X, y = [], []

for i in range(len(words) - seq_length):
    X.append([word_to_idx[w] for w in words[i:i+seq_length]])
    y.append(word_to_idx[words[i+seq_length]])

X = np.array(X)
y = np.array(y)
y_onehot = to_categorical(y, vocab_size)

print("X shape:", X.shape)
print("y shape:", y_onehot.shape)

X shape: (142, 5)
y shape: (142, 103)


In [16]:
model = Sequential()

model.add(Embedding(input_dim=vocab_size, output_dim=64, input_length=seq_length))

model.add(SimpleRNN(128, return_sequences=False))

model.add(Dropout(0.3))

model.add(Dense(vocab_size, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

model.summary()


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_2 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [25]:


early = EarlyStopping(
    monitor='val_loss',
    patience=8,
    min_delta=0.0005,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-5
)

history = model.fit(
    X,
    y_onehot,
    epochs=80,
    batch_size=16,
    validation_split=0.2,
    callbacks=[early, reduce_lr]
)


Epoch 1/80
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.2476 - loss: 4.4745 - val_accuracy: 0.0345 - val_loss: 4.6625 - learning_rate: 1.0000e-05
Epoch 2/80
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.3679 - loss: 4.4540 - val_accuracy: 0.0345 - val_loss: 4.6628 - learning_rate: 1.0000e-05
Epoch 3/80
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.3505 - loss: 4.4525 - val_accuracy: 0.0345 - val_loss: 4.6630 - learning_rate: 1.0000e-05
Epoch 4/80
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.3034 - loss: 4.4646 - val_accuracy: 0.0345 - val_loss: 4.6633 - learning_rate: 1.0000e-05
Epoch 5/80
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.3343 - loss: 4.4602 - val_accuracy: 0.0345 - val_loss: 4.6636 - learning_rate: 1.0000e-05
Epoch 6/80
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.3468 - loss: 4.4494 - val_accuracy: 0.0345 - val_loss: 4.6638 - learning_rate: 1.0000e-05
Epoch 7/80
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.3667 - loss: 4.4547

In [26]:


def sample_with_temperature(preds, temperature=1.0):
    preds = np.asarray(preds).astype('float64')

    preds = np.log(preds + 1e-8) / temperature
    exp_preds = np.exp(preds)
    preds = exp_preds / np.sum(exp_preds)
    return np.random.choice(len(preds), p=preds)

def generate_text(model, seed_text, num_words, seq_length, word_to_idx, idx_to_word, temperature=1.0):
    text = seed_text.lower()
    text = re.sub(r'[^a-zçğıöşü ]+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    words = text.split()

    generated = words.copy()

    for _ in range(num_words):
        current = generated[-seq_length:]
        if len(current) < seq_length:
            current = ['bir'] * (seq_length - len(current)) + current

        try:
            input_seq = [word_to_idx[w] for w in current]
        except:
            input_seq = [word_to_idx['bir']] * seq_length

        preds = model.predict(np.array([input_seq]), verbose=0)[0]


        next_idx = sample_with_temperature(preds, temperature=temperature)
        next_word = idx_to_word[next_idx]

        generated.append(next_word)

    return ' '.join(generated)


In [27]:
print("Temperature = 0.7")
print(generate_text(model, "bir gün dev", 30, seq_length, word_to_idx, idx_to_word, temperature=0.7))

print("\nTemperature = 1.0")
print(generate_text(model, "bir gün dev", 30, seq_length, word_to_idx, idx_to_word, temperature=1.0))

print("\nTemperature = 1.2")
print(generate_text(model, "bir gün dev", 30, seq_length, word_to_idx, idx_to_word, temperature=1.2))


Temperature = 0.7
bir gün dev iken biraz keloğlan kes keloğlana yanına ona cevabı berber ormana geçmiş ben gitmiş kes ve al yokmuş bu almış keloğlana saman sallar oğlum günler çekmemişler kalmadı saman pireler yasak bir

Temperature = 1.0
bir gün dev yaşa ver beğenmiş mıngır tıngır yoksulluk iyileşince daha bana hiç ona zaman yiyecek ormana iyi akıllıymış kalmadı beğenmiş ona pireler bana evvel yanına keserken iyi ama berber demiş dönmüş hiç

Temperature = 1.2
bir gün dev daha de almış hastalanmış keloğlan almış biri yiyecek çocuk almış pireler korkmuş hiç çalışmış ormana çalışmış adında al tıngır getirmiş izin evinde annemin dönmüş çocuk düşmüş altınları ormanda yoksulmuş al


In [ ]:
#Hocam, modelin aşırı öğrenmesini engellemek için RNN katmanının ardından dropout kullandım. Böylece modelin eğitimi sırasında aynı örnekleri ezberleme ihtimali azalmış oldu.
#Bunun yanında gereksiz yere uzun süren epochları durdurmak ve en iyi ağırlıkları korumak için EarlyStopping ekledim. EarlyStopping’in daha kararlı çalışması için de
#eğitim verisinin %20’sini validation setine ayırdım. Son olarak yeni metin üretiminde, daha akıcı cümleler elde etmek ve “ki ki”, “bir bir” gibi tekrarlayan kelimeleri azaltmak için Temperature Sampling kullandım.
#Temperature değerlerini 0.7 / 1.0 / 1.2 olarak deneyip farklı akıcılık seviyelerinde çıktılar ürettim